## Determine number of HTGs from GenBank or RefSeq genomes

In [12]:
import re
import subprocess
from typing import List


def fetch_protein_provenance(
    accession: str,
    pattern: str = r"DBSOURCE|coded_by|BioProject|BioSample|Assembly",
) -> List[str]:
    """
    Fetch a GenPept record from NCBI using Entrez Direct `efetch`
    and return whether record is from a GenBank or RefSeq annotated genomes

    Requires Entrez Direct tools to be installed and `efetch` to be on PATH.
    """

    cmd = [
        "efetch",
        "-db", "protein",
        "-id", accession,
        "-format", "gb",
    ]

    result = subprocess.run(
        cmd,
        check=True,
        text=True,
        capture_output=True,
    )

    regex = re.compile(pattern)

    lines = [
        line
        for line in result.stdout.splitlines()
        if regex.search(line)
    ]
    print(lines)
    for line in lines:
        if 'COMMENT     ##Genome-Assembly-Data-START##' in line or "DBSOURCE    REFSEQ: accession" in line:
            return True
    return False


In [2]:
import pandas as pd
df=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col="accession")
df=df[df.tree_HTG]

In [7]:
for index, row in df[df.NCBI_genome.astype(str)=='nan'].iterrows():
    df.loc[index, 'NCBI_genome']=fetch_protein_provenance(row.sseqid)

In [ ]:
%%bash
# 1. Get all arthropod genome assemblies
curl -L 'https://www.ebi.ac.uk/ena/portal/api/search?result=assembly&query=tax_tree(6656)&fields=accession,scientific_name,tax_id,assembly_name,assembly_level,first_public,last_updated,description&format=tsv&limit=0' \
  > ena_arthropod_assemblies.tsv

# 2. Get all arthropod annotated sequence records
curl -L 'https://www.ebi.ac.uk/ena/portal/api/search?result=sequence&query=tax_tree(6656)&fields=accession,scientific_name,tax_id,description,first_public,last_updated&format=tsv&limit=0' \
  > ena_arthropod_sequences.tsv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    43  100    43    0     0    129      0 --:--:-- --:--:-- --:--:--   129
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed


In [ ]:
import pandas as pd

assemblies = pd.read_csv("ena_arthropod_assemblies.tsv", sep="\t")
seqs = pd.read_csv("ena_arthropod_sequences.tsv", sep="\t")

# One row per arthropod assembly
assemblies = assemblies.drop_duplicates(subset=["accession"])

# One row per species with at least one annotated ENA sequence record
annotated_species = (
    seqs[["tax_id", "scientific_name"]]
    .drop_duplicates()
    .assign(has_ena_sequence_annotation=True)
)

# Mark assembly species that also have annotated sequence records in ENA
annotated_assemblies = assemblies.merge(
    annotated_species,
    on=["tax_id", "scientific_name"],
    how="left"
)

annotated_assemblies["has_ena_sequence_annotation"] = (
    annotated_assemblies["has_ena_sequence_annotation"].fillna(False)
)

annotated_assemblies.to_csv(
    "ena_arthropod_assemblies_with_annotation_flag.tsv",
    sep="\t",
    index=False
)

## Verify that NCBI genome taxa have sequence representation in clusteredNR

In [1]:
import pandas as pd
import subprocess
from Bio import SeqIO
import multiprocessing as mp

In [26]:
import subprocess
def get_rep(tid):
    try:
        a=subprocess.run(['sh','scripts/get_taxid_accessions.sh',str(tid)], capture_output=True,text=True,check=True)
    
        a=[x.strip() for x in a.stdout.split("\n--------------------\n")[1].split("\n")]
    except:
 
        a='None'
    return a


In [18]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_order_class_taxid(taxid):
    """
    Return the family taxid for a given NCBI taxid.

    Parameters
    ----------
    taxid : int

    Returns
    -------
    order, class
    """
    lineage = ncbi.get_lineage(taxid)
    ranks = ncbi.get_rank(lineage)
  

    order_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "order"),
        None
    )
    


    order_name = ncbi.get_taxid_translator([order_taxid])[order_taxid]

    family_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "family"),
        None
    )
    family_name = ncbi.get_taxid_translator([family_taxid])[family_taxid]

    
    class_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "class"),
        None
    )

    class_name = ncbi.get_taxid_translator([class_taxid])[class_taxid]
    return family_name, order_name, class_name

In [7]:
import pandas as pd
df=pd.read_csv("datasets/annotated_genomes_ncbi_5_27_26.tsv",sep="\t")

In [20]:
for index, row in df.iterrows():
    tid=row['Organism Taxonomic ID']
    try:
        df.loc[index,['family', 'order','class']]=get_order_class_taxid(tid)
    except:
        print(tid,index)

In [23]:
df.to_csv("datasets/annotated_genomes_ncbi_5_27_26.tsv",sep="\t")

In [25]:
df=pd.read_csv("datasets/annotated_genomes_ncbi_5_27_26.tsv",sep="\t",index_col=0)

In [1]:
import multiprocessing as mp

In [9]:
import subprocess
def get_rep(tid):
    print(tid)
    try:
        a=subprocess.run(['sh','scripts/get_taxid_accessions.sh',str(tid)], capture_output=True,text=True,check=True)
    
        a= a.stdout.split("\n--------------------\n")[1].split("\n")
        n=len(a)
        del a
        return n
    except:
        print("fail",tid)
 
        a='None'
    return a


In [5]:
get_rep(644536)

13435

In [10]:
td=list(df['Organism Taxonomic ID'])
pool = mp.Pool(processes=20)
n_hits=pool.map(get_rep, td)

47313066895165511333430195285846954166361481309
139456292122369319



35570





1421715644536
1390455038794029





13037
315576
7200
345580
186751
680683
150431
30033
13686
95512
170843
43151
543639
139649
129105
375433
248454
6943
42839
13191
83912
1035111
7091
217165
189513
34621
27457
64459
80765
7226
328185
1163345
684658
197152
171585
7373
7245
6832
201502
136180
29058
178769
85310
7108
7116
7240
1496333
72248
197161
307658
7009
7167
334116
171605
6978
28588
684658
543639
91739
51811
417401
6669
286306
103775
82600
286459
1163346
116150
76193
7241
392227
6832
185578
6728
168631
995702
42275
29030
7165
89055
337810
59916
125945
7111
32260
103775
1431903
58733
1820382
1486046
7173
286306
2705178
7067
29057
159736
7260
319348
6661
7291
115357
115081
7227
7141
35525
174621
30069
191418
41139
444603
7238
85661
7445
34632
7176
2015173
34620
53552
43146
6941
7141
143995
290404
2795564
614101
30066
40932
50390
7227
6661
115077
42026
72036
129554
78633
1518534
124917
1437191
202808
26048

In [27]:
td_map={x:y for x,y in zip(td,n_hits)}
df=df.set_index('Organism Taxonomic ID')
for t in td_map:
    df.loc[t, 'n_hits']=td_map[t]


In [29]:
for o,dfi in df.groupby('order'):
    print(o, dfi.n_hits.sum())

Amphipoda 103952.0
Anostraca 58690.0
Araneae 1458114.0
Balanomorpha 73808.0
Blattodea 253910.0
Calanoida 44331.0
Coleoptera 1446155.0
Decapoda 949457.0
Diplostraca 284325.0
Diptera 2225837.0
Entomobryomorpha 129176.0
Ephemeroptera 84463.0
Euphausiacea 39889.0
Harpacticoida 46288.0
Hemiptera 1071930.0
Hymenoptera 1781440.0
Isopoda 85709.0
Ixodida 1433729.0
Lepidoptera 1541756.0
Mesostigmata 40242.0
Neuroptera 13007.0
Odonata 45033.0
Orthoptera 108746.0
Pantopoda 21596.0
Phasmatodea 63909.0
Podocopida 43967.0
Pollicipedomorpha 17620.0
Polydesmida 14537.0
Pseudoscorpiones 23843.0
Psocodea 60296.0
Sarcoptiformes 359713.0
Scolopendromorpha 17227.0
Scorpiones 34554.0
Scutigeromorpha 18340.0
Siphonaptera 15310.0
Siphonostomatoida 100073.0
Spirostreptida 11683.0
Stomatopoda 21652.0
Symphypleona 47874.0
Thysanoptera 96890.0
Trichoptera 13217.0
Trombidiformes 75189.0
Xiphosura 51557.0
Zygentoma 46919.0


In [32]:
under_orders=set(df[df.index.isin([x for x,y in zip(td,n_hits) if y<1000])].order)

In [33]:
under_orders

{'Coleoptera', 'Diptera', 'Ixodida', 'Psocodea'}